# Titanic — Feature Engineering
**Goal:** Transform raw data into meaningful features that a Machine Learning model can understand.

## 1. Load Cleaned Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('Titanic-Dataset.csv') # Loading raw to get Name column back
print('Shape:', df.shape)

## 2. Extract Title from Name

In [ ]:
df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\\.', expand=False)
df['Title'] = df['Title'].replace(['Lady', 'Countess','Capt', 'Col','Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Other')
df['Title'] = df['Title'].replace('Mlle', 'Miss').replace('Ms', 'Miss').replace('Mme', 'Mrs')
print(df['Title'].value_counts())

## 3. Family Features

In [ ]:
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
df[['FamilySize', 'IsAlone']].head()

## 4. Age Binning & Fare Transformation

In [ ]:
df['Age'] = df.groupby(['Pclass', 'Title'])['Age'].transform(lambda x: x.fillna(x.median()))
df['AgeBin'] = pd.cut(df['Age'], bins=[0, 12, 18, 35, 60, 100], labels=['Child', 'Teen', 'Adult', 'Middle-Aged', 'Senior'])
df['LogFare'] = np.log1p(df['Fare'])
df[['AgeBin', 'LogFare']].head()

## 5. Encoding Categorical Data

In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})
df['AgeBin'] = le.fit_transform(df['AgeBin'])

# One-Hot encode Embarked and Title
df = pd.get_dummies(df, columns=['Embarked', 'Title'], drop_first=True)

# Convert bool to int
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

## 6. Final Cleanup & Save

In [ ]:
cols_to_drop = ['PassengerId', 'Name', 'Ticket', 'Cabin', 'SibSp', 'Parch', 'Fare']
df.drop(columns=cols_to_drop, inplace=True)
df.to_csv('Titanic-Featured.csv', index=False)
print('Saved Featured Data. Shape:', df.shape)
df.head()